In [ ]:
# =====================================================
# PROJET :
# PRÉDICTION DU DIABÈTE
# =====================================================

"""
OBJECTIF

Construire un modèle de Machine Learning capable
de prédire si une personne est diabétique.

Type de problème :
Classification Binaire

0 = Non diabétique
1 = Diabétique

Pipeline ML utilisé :

1. Chargement des données
2. Analyse exploratoire
3. Prétraitement
4. Train/Test Split
5. Pipeline
6. Entraînement
7. Prédiction
8. Évaluation
"""

# =====================================================
# 1. IMPORTATION DES LIBRAIRIES
# =====================================================

import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

# =====================================================
# 2. CHARGEMENT DES DONNÉES
# =====================================================

df = pd.read_csv("diabetes_prediction_dataset.csv")

print("Premières lignes du dataset")
print(df.head())

print("\nDimensions du dataset")
print(df.shape)

"""
Pourquoi ?

Toujours vérifier :

- les colonnes
- les types
- les dimensions

avant de construire un modèle.
"""

# =====================================================
# 3. ANALYSE DE LA VARIABLE CIBLE
# =====================================================

print("\nRépartition des classes")

print(df["diabetes"].value_counts())

print("\nRépartition (%)")

print(
    df["diabetes"].value_counts(normalize=True) * 100
)

"""
Pourquoi ?

Vérifier si les classes sont équilibrées.

Exemple :

90% Non diabétique
10% Diabétique

Si le dataset est déséquilibré :

Accuracy seule devient trompeuse.

Nous utiliserons donc :

- Precision
- Recall
- F1-Score
- ROC-AUC
"""

# =====================================================
# 4. SÉPARATION FEATURES / TARGET
# =====================================================

X = df.drop("diabetes", axis=1)

y = df["diabetes"]

"""
X = Variables explicatives

y = Variable cible

Le modèle apprendra :

X  --->  y
"""

# =====================================================
# 5. IDENTIFICATION DES TYPES DE VARIABLES
# =====================================================

cat_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

num_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("\nColonnes catégorielles")
print(cat_cols)

print("\nColonnes numériques")
print(num_cols)

"""
Pourquoi ?

Les variables ne sont pas traitées
de la même manière.

Numériques :

- age
- bmi
- glucose

Catégorielles :

- gender
- smoking_history
"""

# =====================================================
# 6. PREPROCESSING
# =====================================================

preprocess = ColumnTransformer([

    (
        "num",
        StandardScaler(),
        num_cols
    ),

    (
        "cat",
        OneHotEncoder(
            handle_unknown="ignore"
        ),
        cat_cols
    )

])

"""
ColumnTransformer

Permet d'appliquer :

StandardScaler
aux colonnes numériques

et

OneHotEncoder
aux colonnes catégorielles

------------------------------------------------

handle_unknown='ignore'

Si une nouvelle catégorie apparaît
pendant la prédiction :

Train :

Male
Female

Production :

Other

Le modèle ne plante pas.
"""

# =====================================================
# 7. TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

"""
Pourquoi ?

Train :
80%

Test :
20%

Le modèle apprend sur Train.

Le modèle est évalué sur Test.

------------------------------------------------

stratify=y

Permet de conserver
les mêmes proportions de classes.

Exemple :

90% Non diabétique
10% Diabétique

Train :
90% / 10%

Test :
90% / 10%
"""

# =====================================================
# 8. CONSTRUCTION DU PIPELINE
# =====================================================

model = Pipeline([

    (
        "preprocessing",
        preprocess
    ),

    (
        "classifier",
        LogisticRegression()
    )

])

"""
Pipeline

Garantit l'ordre suivant :

1. Prétraitement
2. Transformation
3. Modèle

Sans intervention manuelle.

Très utile pour :

- éviter les erreurs
- déployer le modèle
"""

# =====================================================
# 9. ENTRAINEMENT
# =====================================================

model.fit(

    X_train,
    y_train

)

"""
Le modèle apprend les relations
entre les variables et le diabète.

Exemples :

- âge
- IMC
- glycémie
- HbA1c

et leur influence sur la variable cible.
"""

# =====================================================
# 10. PRÉDICTIONS
# =====================================================

y_pred = model.predict(X_test)

"""
Prédictions finales :

0 = Non diabétique

1 = Diabétique
"""

# Probabilités nécessaires pour ROC-AUC

y_proba = model.predict_proba(X_test)[:, 1]

"""
predict_proba()

Retourne la probabilité
d'être diabétique.

Exemple :

0.95 = 95%

0.20 = 20%
"""

# =====================================================
# 11. ÉVALUATION DU MODÈLE
# =====================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

roc_auc = roc_auc_score(
    y_test,
    y_proba
)

"""
Pourquoi plusieurs métriques ?

Aucune métrique n'est parfaite.

Nous calculons :

Accuracy
Precision
Recall
F1
ROC-AUC

pour avoir une vue complète
des performances.
"""

# =====================================================
# 12. AFFICHAGE DES MÉTRIQUES
# =====================================================

print("\n========== METRICS ==========\n")

print(f"Accuracy  : {accuracy:.4f}")

print(f"Precision : {precision:.4f}")

print(f"Recall    : {recall:.4f}")

print(f"F1-Score  : {f1:.4f}")

print(f"ROC-AUC   : {roc_auc:.4f}")

# =====================================================
# 12 BIS. INTERPRÉTATION DES RÉSULTATS OBTENUS
# =====================================================

print("\n========== ANALYSE DU MODELE ==========\n")

print(f"""
Accuracy : {accuracy:.2%}

Question :
Combien de prédictions sont correctes ?

Réponse :
Le modèle réalise {accuracy:.2%} de bonnes prédictions.

Interprétation :
Le modèle semble très performant globalement.

Attention :
L'Accuracy seule ne suffit pas lorsque les
classes sont déséquilibrées.
""")

print(f"""
------------------------------------------------

Precision : {precision:.2%}

Question :
Quand le modèle prédit 'Diabétique',
a-t-il raison ?

Réponse :
Oui dans {precision:.2%} des cas.

Interprétation :
Le modèle génère relativement peu
de fausses alertes.

Exemple :

Sur 100 patients déclarés diabétiques
par le modèle :

≈ {precision*100:.0f} sont réellement diabétiques
≈ {(1-precision)*100:.0f} ne le sont pas

La Precision est bonne.
""")

print(f"""
------------------------------------------------

Recall : {recall:.2%}

Question :
Parmi tous les vrais diabétiques,
combien ont été détectés ?

Réponse :
Le modèle détecte {recall:.2%}
des diabétiques.

Interprétation :

Sur 100 personnes réellement diabétiques :

≈ {recall*100:.0f} sont détectées
≈ {(1-recall)*100:.0f} sont manquées

Pour un contexte médical,
ce résultat peut encore être amélioré.

Le modèle manque encore
plusieurs patients diabétiques.
""")

print(f"""
------------------------------------------------

F1-Score : {f1:.2%}

Question :
Quel est l'équilibre entre
Precision et Recall ?

Réponse :
Le compromis global est de {f1:.2%}.

Interprétation :

Le modèle possède :

- une bonne Precision
- un Recall moyen

Le Recall tire le F1-score vers le bas.

Une amélioration du Recall
augmenterait naturellement le F1-score.
""")

print(f"""
------------------------------------------------

ROC-AUC : {roc_auc:.2%}

Question :
Le modèle sait-il distinguer
les diabétiques des non diabétiques ?

Réponse :
Oui.

Interprétation :

Avec un ROC-AUC de {roc_auc:.2%},
le modèle sépare très bien
les deux classes.

Valeurs de référence :

0.50 = Aléatoire
0.70 = Correct
0.80 = Bon
0.90 = Excellent

Votre modèle est excellent
sur ce critère.
""")

# =====================================================
# INTERPRÉTATION PLUS SIMPLE
# =====================================================

print("\n========== INTERPRETATION ==========\n")

print(f"""
Accuracy = {accuracy:.2%}

{accuracy:.2%} des prédictions sont correctes.
""")

print(f"""
Precision = {precision:.2%}

Quand le modèle dit "diabétique",
il a raison {precision:.2%} du temps.
""")

print(f"""
Recall = {recall:.2%}

Parmi tous les vrais diabétiques,
le modèle en détecte {recall:.2%}.
""")

print(f"""
F1-Score = {f1:.2%}

Le compromis global entre Precision
et Recall est de {f1:.2%}.
""")

print(f"""
ROC-AUC = {roc_auc:.2%}

Le modèle sépare les diabétiques
des non-diabétiques avec une performance
de {roc_auc:.2%}.
""")

# =====================================================
# 13. CONFUSION MATRIX
# =====================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\n========== CONFUSION MATRIX ==========\n")

print(cm)

"""
Confusion Matrix

                Réel

             0      1

Prédit 0    TN     FN

Prédit 1    FP     TP

------------------------------------------------

TN

Vrai négatif

------------------------------------------------

TP

Vrai positif

------------------------------------------------

FP

Faux positif

------------------------------------------------

FN

Faux négatif

Dans le domaine médical,
FN est souvent l'erreur
la plus dangereuse.
"""

# =====================================================
# 14. CLASSIFICATION REPORT
# =====================================================

print("\n========== CLASSIFICATION REPORT ==========\n")

print(
    classification_report(
        y_test,
        y_pred
    )
)

"""
Le classification_report affiche :

- Precision
- Recall
- F1-score

pour chaque classe.

C'est souvent la première chose
qu'un Data Scientist regarde.
"""

# =====================================================
# 15. CHOIX DE LA MÉTRIQUE MÉTIER
# =====================================================

print(
"""
=================================================

QUELLE MÉTRIQUE CHOISIR ?

Accuracy

-> Dataset équilibré

------------------------------------------------

Recall

-> Santé
-> Diabète
-> Cancer
-> Fraude

Objectif :
Ne pas rater de cas

------------------------------------------------

Precision

-> Spam
-> Justice
-> Validation documentaire

Objectif :
Limiter les fausses alertes

------------------------------------------------

F1-Score

-> Dataset déséquilibré

Objectif :
Trouver un compromis

------------------------------------------------

ROC-AUC

-> Comparer plusieurs modèles

Objectif :
Mesurer la capacité de séparation

=================================================
"""
)

# =====================================================
# 16. CONCLUSION
# =====================================================

print(
"""
Pipeline ML réalisé :

✓ Analyse des données

✓ Prétraitement

✓ Encodage

✓ Standardisation

✓ Train/Test Split

✓ Pipeline

✓ Logistic Regression

✓ Accuracy

✓ Precision

✓ Recall

✓ F1-Score

✓ ROC-AUC

✓ Confusion Matrix

✓ Classification Report

PROCHAINE ÉTAPE :

Comprendre les hyperparamètres :

- C
- penalty
- solver
- max_iter

Puis :

- Cross Validation
- GridSearchCV
- Optimisation du modèle
"""
)